In [32]:
import pandas as pd
import numpy as np
df = pd.read_csv('Employee_Dataset.csv')
df.head()

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active
0,EMP1000,IT,Senior Analyst,60,85000,invalid,01/04/2021,5,3,NaN
1,emp_1,IT,Analyst,NaN,55000,10/06/2020,invalid,5,3,True
2,EMP1002,Sales,NaN,28,85000,NaN,2022-03-01,3,4,NaN
3,EMP1003,it,Analyst,45,35000,invalid,01/04/2021,1,3,NaN
4,NaN,IT,mgr,150,85000,NaN,invalid,12,2,True


In [33]:

# =============================
# DATA CLEANING & VALIDATION
# =============================

# 1. Convert joining_date
df['joining_date_clean'] = pd.to_datetime(df['joining_date'], errors='coerce')
failed_joining_date = df['joining_date_clean'].isna().sum()



C:\Users\Sathvik\AppData\Local\Temp\ipykernel_177568\455481182.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['joining_date_clean'] = pd.to_datetime(df['joining_date'], errors='coerce')


In [34]:
# 2. Clean employee_id
df['employee_id_clean'] = df['employee_id'].astype(str).str.strip().str.upper()
duplicate_employees = df['employee_id_clean'].duplicated().sum()

In [35]:

# 3. Standardize department & salary
df['department_clean'] = df['department'].str.strip().str.title()
df['salary_clean'] = pd.to_numeric(df['salary'], errors='coerce')
avg_salary_dept = df.groupby('department_clean')['salary_clean'].mean()

In [36]:
df['age_clean'] = pd.to_numeric(df['age'], errors='coerce')
invalid_age_valid_salary = df[(df['salary_clean'].notna()) & (df['age_clean'].isna())]

# 5. Salary outliers (IQR)

In [37]:
Q1 = df['salary_clean'].quantile(0.25)
Q3 = df['salary_clean'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
salary_outliers = df[(df['salary_clean'] < lower) | (df['salary_clean'] > upper)]


In [38]:
# 6. Performance rating median
df['performance_rating_clean'] = pd.to_numeric(df['performance_rating'], errors='coerce')
df['designation_clean'] = df['designation'].str.strip().str.title()
median_rating = df.groupby('designation_clean')['performance_rating_clean'].median()

In [39]:
# 7. Invalid promotion dates
df['last_promotion_date_clean'] = pd.to_datetime(df['last_promotion_date'], errors='coerce')
invalid_promotion = df[df['last_promotion_date_clean'] < df['joining_date_clean']]

In [40]:



# 8. Experience mismatch
df['experience_clean'] = pd.to_numeric(df['experience_years'], errors='coerce')
experience_age_mismatch = df[df['experience_clean'] > df['age_clean']]

In [41]:
# 9. Active employees count
df['is_active_clean'] = df['is_active'].astype(str).str.lower().map({'true':True,'false':False})
active_count = df[df['is_active_clean'] == True].groupby('designation_clean').size()

In [42]:

# 10. Inactive with recent promotions
recent_cutoff = pd.Timestamp.today() - pd.DateOffset(years=2)
inactive_recent_promo = df[(df['is_active_clean'] == False) & 
                           (df['last_promotion_date_clean'] >= recent_cutoff)]

In [43]:

# 11. Tenure calculation
df['tenure_years'] = (pd.Timestamp.today() - df['joining_date_clean']).dt.days / 365
p90 = df['tenure_years'].quantile(0.9)
high_tenure = df[df['tenure_years'] > p90]

In [44]:
# 12. Dept >25% missing salary
salary_missing_pct = df.groupby('department_clean')['salary_clean'].apply(lambda x: x.isna().mean())
dept_high_missing = salary_missing_pct[salary_missing_pct > 0.25]

In [45]:
# 13. High performance but low salary
median_salary = df['salary_clean'].median()
df['high_perf_low_salary_flag'] = np.where(
    (df['performance_rating_clean'] >= 4) & 
    (df['salary_clean'] < median_salary), 1, 0)
print(df['high_perf_low_salary_flag'].value_counts())

high_perf_low_salary_flag
0    942
1     58
Name: count, dtype: int64


In [46]:

# 14. No promotion but >5 years exp
no_promo_high_exp = df[(df['last_promotion_date_clean'].isna()) & 
                       (df['experience_clean'] > 5)]

In [49]:
# 15. Multi-constraint violation
df['constraint1'] = df['joining_date_clean'].isna()
df['constraint2'] = df['salary_clean'].isna()
df['constraint3'] = df['age_clean'].isna()
df['violation_count'] = df[['constraint1','constraint2','constraint3']].sum(axis=1)
df['multi_violation_flag'] = df['violation_count'] >= 2
df

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,is_active_clean,tenure_years,high_perf_low_salary_flag,constraint1,constraint2,constraint3,violation_count,multi_violation_flag,promotion_gap,salary_exp_ratio
0,EMP1000,IT,Senior Analyst,60,85000,invalid,01/04/2021,5,3,NaN,...,NaN,NaN,0,True,False,False,1,False,NaN,17000.000000
1,emp_1,IT,Analyst,NaN,55000,10/06/2020,invalid,5,3,True,...,True,5.353425,0,False,False,True,1,False,NaN,11000.000000
2,EMP1002,Sales,NaN,28,85000,NaN,2022-03-01,3,4,NaN,...,NaN,NaN,0,True,False,False,1,False,NaN,28333.333333
3,EMP1003,it,Analyst,45,35000,invalid,01/04/2021,1,3,NaN,...,NaN,NaN,0,True,False,False,1,False,NaN,35000.000000
4,NaN,IT,mgr,150,85000,NaN,invalid,12,2,True,...,True,NaN,0,True,False,False,1,False,NaN,7083.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EMP1995,Finance,NaN,60,55000,2019-05-10,2022-03-01,1,4,yes,...,NaN,6.764384,1,False,False,False,0,False,NaN,55000.000000
996,EMP1996,IT,Analyst,35,250000,invalid,invalid,-2,5,NaN,...,NaN,NaN,0,True,False,False,1,False,NaN,-125000.000000
997,EMP1997,Sales,Senior Analyst,150,250000,2021/07/15,invalid,3,excellent,yes,...,NaN,4.580822,0,False,False,False,0,False,NaN,83333.333333
998,emp_998,Finance,NaN,60,85000,2019-05-10,NaN,12,NaN,True,...,True,6.764384,0,False,False,False,0,False,NaN,7083.333333


In [50]:
df['last_promotion_date_std'] = pd.to_datetime(df['last_promotion_date'], errors='coerce', dayfirst=True)

df['joining_date_std'] = pd.to_datetime(df['joining_date'],errors='coerce',dayfirst=True)

promotion_gap = df[(df['last_promotion_date_std'] - df['joining_date_std']).dt.days/365.25 < 1]
promotion_gap

C:\Users\Sathvik\AppData\Local\Temp\ipykernel_177568\3329214143.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['joining_date_std'] = pd.to_datetime(df['joining_date'],errors='coerce',dayfirst=True)


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,high_perf_low_salary_flag,constraint1,constraint2,constraint3,violation_count,multi_violation_flag,promotion_gap,salary_exp_ratio,last_promotion_date_std,joining_date_std
9,EMP1009,it,NaN,22,120000,10/06/2020,01/04/2021,NaN,4,NaN,...,0,False,False,False,0,False,0.246575,NaN,2021-04-01,2020-06-10
24,EMP1024,Finance,mgr,35,85000,2021/07/15,01/04/2021,8,3,True,...,0,False,False,False,0,False,-0.526027,10625.000000,2021-04-01,2021-07-15
40,emp_40,NaN,NaN,60,85000,2021/07/15,01/04/2021,12,5,True,...,0,False,False,False,0,False,-0.526027,7083.333333,2021-04-01,2021-07-15
44,emp_44,sales,mgr,45,₹75000,2021/07/15,01/04/2021,8,2,True,...,0,False,True,False,1,False,-0.526027,NaN,2021-04-01,2021-07-15
47,NaN,Finance,Analyst,unknown,250000,2021/07/15,01/04/2021,3,excellent,True,...,0,False,False,True,1,False,-0.526027,83333.333333,2021-04-01,2021-07-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,EMP1943,Finance,mgr,45,35000,10/06/2020,01/04/2021,-2,NaN,NaN,...,0,False,False,False,0,False,0.246575,-17500.000000,2021-04-01,2020-06-10
945,NaN,HR,Manager,-5,35000,10/06/2020,01/04/2021,3,NaN,True,...,0,False,False,False,0,False,0.246575,11666.666667,2021-04-01,2020-06-10
966,NaN,it,Analyst,45,NaN,10/06/2020,01/04/2021,1,NaN,no,...,0,False,True,False,1,False,0.246575,NaN,2021-04-01,2020-06-10
974,emp_974,NaN,mgr,unknown,85000,2021/07/15,01/04/2021,3,4,False,...,0,False,False,True,1,False,-0.526027,28333.333333,2021-04-01,2021-07-15


In [ ]:
df['sal_std'] = (df['salary'].astype(str).str.replace('₹', '', regex=False).str.replace(',', '', regex=False))
df['sal_std'] = pd.to_numeric(df['sal_std'], errors='coerce')
df['dept_std'] = df['department'].astype(str).str.strip().str.lower()
df['performance_rating_num'] = pd.to_numeric(df['performance_rating'], errors='coerce')
df['dept_avg_salary'] = (df.groupby('dept_std')['sal_std'].transform('mean'))
df['dept_median_perf'] = (df.groupby('dept_std')['performance_rating_num'].transform('median'))
result = df[(df['sal_std'] > df['dept_avg_salary']) & (df['performance_rating_num'] < df['dept_median_perf'])]
result

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,multi_violation_flag,promotion_gap,salary_exp_ratio,last_promotion_date_std,joining_date_std,sal_std,dept_std,performance_rating_num,dept_avg_salary,dept_median_perf
45,NaN,Finance,NaN,NaN,120000,NaN,01/04/2021,1,1,yes,...,True,NaN,120000.000000,2021-04-01,NaT,120000.0,finance,1.0,107397.959184,3.0
89,emp_89,NaN,Senior Analyst,22,120000,10/06/2020,NaN,1,1,no,...,False,NaN,120000.000000,NaT,2020-06-10,120000.0,nan,1.0,96039.603960,3.0
115,emp_115,sales,ANALYST,28,250000,2021/07/15,invalid,-2,2,no,...,False,NaN,-125000.000000,NaT,2021-07-15,250000.0,sales,2.0,110842.696629,3.0
127,NaN,Finance,NaN,150,120000,invalid,2022-03-01,3,1,True,...,False,NaN,40000.000000,NaT,NaT,120000.0,finance,1.0,107397.959184,3.0
157,NaN,Finance,ANALYST,22,250000,2019-05-10,2022-03-01,3,1,False,...,False,NaN,83333.333333,NaT,2019-05-10,250000.0,finance,1.0,107397.959184,3.0
169,emp_169,it,mgr,60,120000,2021/07/15,01/04/2021,NaN,2,NaN,...,False,-0.526027,NaN,2021-04-01,2021-07-15,120000.0,it,2.0,103373.205742,3.0
173,emp_173,Finance,Manager,45,120000,10/06/2020,invalid,8,1,False,...,False,NaN,15000.000000,NaT,2020-06-10,120000.0,finance,1.0,107397.959184,3.0
216,emp_216,NaN,Senior Analyst,60,120000,invalid,invalid,8,2,NaN,...,False,NaN,15000.000000,NaT,NaT,120000.0,nan,2.0,96039.603960,3.0
217,NaN,it,NaN,28,250000,2021/07/15,2022-03-01,NaN,2,yes,...,False,NaN,NaN,NaT,2021-07-15,250000.0,it,2.0,103373.205742,3.0
223,NaN,HR,Manager,NaN,120000,2019-05-10,NaN,-2,1,True,...,False,NaN,-60000.000000,NaT,2019-05-10,120000.0,hr,1.0,110117.647059,3.0
